# LAB- P9: El Modelo de Crecimiento Óptimo de Ramsey
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de LAB-**: LAB-P9-v1.0
*   **Capítulo de Referencia**: Capítulo 10, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Resolver e interactuar con el modelo canónico de crecimiento óptimo de Ramsey-Cass-Koopmans en tiempo discreto, calculando las condiciones de estabilidad de punto de silla, los autovalores jacobianos, la condición de salto de la variable forward-looking (Consumo) y contrastando la solución linealizada frente a la solución no lineal exacta.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** la microfundamentación intertemporal del crecimiento óptimo bajo vida infinita (enfoque de dinastías familiares).
2.  **Derivar** y entender la Regla de Keynes-Ramsey en tiempo discreto y cómo gobierna las decisiones de consumo-ahorro.
3.  **Analizar** la estabilidad del sistema dinámico mediante la descomposición en autovalores, comprobando que se cumple la condición de punto de silla.
4.  **Calcular** analíticamente el coeficiente de salto ($\theta$) de la variable flexible (consumo) ante perturbaciones estructurales.
5.  **Simular** y comparar el impacto dinámico de shocks tecnológicos (TFP) y de preferencias (paciencia/tasa de descuento $\beta$).
6.  **Evaluar** la precisión del resolvedor linealizado de Blanchard-Khan frente a la solución numérica no lineal exacta (`fsolve`) ante shocks de distinta envergadura.
 (Julia)

> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

### 🕹️ GUÍA RÁPIDA PARA DUMMIES - Modelo de Ramsey-Cass-Koopmans
*   **¿Qué estamos haciendo aquí?** Es como el modelo de Solow, pero aquí la tasa de ahorro no es fija; la gente la elige libremente para maximizar su felicidad a lo largo de las generaciones.
*   **Senda Estable:** El modelo calcula la única trayectoria (saddle path) que evita que la economía se quede sin capital o acumule máquinas inútiles.
*   **¡Prueba esto!** Aplica un shock de productividad (TFP) y mira cómo las familias ajustan instantáneamente su consumo actual para situarse en la nueva senda de crecimiento estable.


In [ ]:
# En Google Colab se activarían y descargarían los paquetes necesarios.
# using Pkg; Pkg.activate("."); Pkg.instantiate()


In [ ]:
using Pkg
Pkg.activate("../..")

using MacroAIComp
using Plots
import Plots: mm
using LinearAlgebra
using NLsolve
using Interact
using BenchmarkTools


## 1. El Modelo de Crecimiento Óptimo de Ramsey

El modelo de Ramsey-Cass-Koopmans microfunda las decisiones de ahorro. A diferencia del modelo de Solow-Swan, donde el ahorro es una fracción fija exógena, aquí el ahorro se determina endógenamente a partir del problema de optimización intertemporal de los hogares, que maximizan su utilidad a lo largo de un horizonte infinito (representando a la dinastía familiar).

### 1.1 El Problema de Optimización del Hogar
El hogar representativo maximiza su utilidad descontada sujeta a la dinámica demográfica (la población crece a tasa constante $n$):
$$\max_{\{c_t\}} \sum_{t=0}^{\infty} \left( \frac{1+n}{1+\theta} \right)^t \ln(c_t)$$
Sujeto a la restricción presupuestaria per cápita:
$$c_t + (1+n)k_{t+1} = w_t + (R_t + 1 - \delta)k_t$$
donde $\theta$ es la tasa de preferencia intertemporal ($1+\theta = 1/\beta$). La condición de primer orden para este problema da lugar a la **Regla de Keynes-Ramsey**:
$$c_{t+1} = \beta (R_{t+1} + 1 - \delta) c_t$$

### 1.2 El Problema de la Empresa y Equilibrio
La empresa maximiza beneficios periodos a periodo bajo una función de producción Cobb-Douglas $y_t = A_t k_t^\alpha$, lo que determina las demandas de factores competitivas:
$$R_t = \alpha A_t k_t^{\alpha-1}, \quad w_t = (1-\alpha)y_t$$

Sustituyendo estas demandas en las ecuaciones del hogar, el equilibrio competitivo dinámico se reduce a un sistema bidimensional de dos ecuaciones en diferencias no lineales:
1.  **Dinámica de Consumo (Regla de Euler):**
    $$c_{t+1} = \beta \left( \alpha A_{t+1} k_{t+1}^{\alpha-1} + 1 - \delta \right) c_t$$
2.  **Dinámica de Acumulación de Capital:**
    $$(1+n)k_{t+1} = (1-\delta)k_t + A_t k_t^\alpha - c_t$$

### 1.3 Estado Estacionario
Eliminando los subíndices de tiempo en el sistema, obtenemos los valores de largo plazo:
$$\bar{k} = \left( \frac{1 - \beta + \beta\delta}{\alpha A\beta} \right)^{\frac{1}{\alpha - 1}}$$
$$\bar{y} = A \bar{k}^\alpha, \quad \bar{c} = \bar{y} - (n + \delta)\bar{k}, \quad \bar{i} = (n + \delta)\bar{k}, \quad \bar{R} = \frac{1}{\beta} - 1 + \delta$$


In [ ]:
params_base = default_calibration(RamseyParams)
ss = compute_ramsey_steady_state(params_base)

println("VALORES DE EQUILIBRIO (SS):")
println("  Capital por trabajador efectivo (K*) : ", round(ss["K"], digits=4))
println("  Consumo (C*)                         : ", round(ss["C"], digits=4))
println("  Producción (Y*)                      : ", round(ss["Y"], digits=4))
println("  Inversión (I*)                       : ", round(ss["I"], digits=4))


## 2. Linealización de Blanchard-Khan y la Condición de Salto

El sistema dinámico no lineal anterior se aproxima linealmente alrededor de su estado estacionario en log-desviaciones ($\hat{c}_t = \ln(c_t/\bar{c})$ y $\hat{k}_t = \ln(k_t/\bar{k})$). El sistema resultante de ecuaciones en diferencias es:
$$\begin{bmatrix} \Delta \hat{c}_t \\ \Delta \hat{k}_t \end{bmatrix} = \begin{bmatrix} J_{11} & J_{12} \\ J_{21} & J_{22} \end{bmatrix} \begin{bmatrix} \hat{c}_t \\ \hat{k}_t \end{bmatrix}$$

donde:
$$J_{11} = - \frac{(\alpha - 1)\Omega\Gamma}{\alpha\beta(1+n)}, \quad J_{12} = \frac{(\alpha - 1)\Omega}{\beta(1+n)}$$
$$J_{21} = - \frac{\Gamma}{\alpha\beta(1+n)}, \quad J_{22} = \frac{1 - \beta(1+n)}{\beta(1+n)}$$
siendo $\Omega \equiv 1-\beta+\beta\delta$ y $\Gamma \equiv 1-\beta+\beta\delta - \alpha\beta(\delta+n)$.

Este sistema tiene una estructura de **punto de silla** (un autovalor estable y otro inestable). Al producirse un shock, la variable rígida (Capital, $\hat{k}_t$) no puede saltar instantáneamente. Es la variable flexible (Consumo, $\hat{c}_t$) la que realiza un **salto discreto instantáneo** en el periodo de impacto para colocarse sobre la trayectoria estable:
$$\hat{c}_0 = \theta \hat{k}_0 \quad \text{donde } \theta = \frac{\alpha(\alpha - 1)\Omega}{(\alpha - 1)\Omega\Gamma + \alpha\beta(1 + n)\lambda_1}$$
donde $\lambda_1$ es el autovalor estable del Jacobiano $J$.


In [ ]:
println("Autovalores de la matriz de transición (Blanchard-Kahn):")
J, lambda_1, lambda_2, theta = compute_ramsey_transition_matrix(params_base)
println(eigvals(J))


## 3. Simulación Interactiva: Transición ante Shocks de Productividad y Preferencias

A continuación, simularemos la economía partiendo del estado estacionario calibrado en el libro ($\alpha=0.35, \beta=0.97, \delta=0.06, n=0.02$). En el período $t = 5$, ocurre un shock estructural permanente. 

Puedes mover el control interactivo para simular un shock de productividad ($A_1$) o de paciencia ($\beta_1$), y observar el ajuste en tiempo real.


In [ ]:
# Simulación interactiva: Shock permanente Ramsey
@manipulate for A_final in 0.90:0.01:1.20, beta_final in 0.92:0.01:0.99
    
    params_init = default_calibration(RamseyParams)
    ss_init = compute_ramsey_steady_state(params_init)
    K0 = ss_init["K"]
    T_sim = 80
    t_shock = 5
    
    res = solve_ramsey_linearized(params_init, K0, A_final, params_init.n, beta_final, T_sim, t_shock)
    
    params_fin = RamseyParams(params_init.alpha, beta_final, params_init.delta, params_init.n, A_final)
    ss_fin = compute_ramsey_steady_state(params_fin)
    
    t_axis = 0:(T_sim - 1)
    
    p1 = plot(t_axis, res["Y"], color=:blue, lw=2.5, label="Renta (Y)")
    hline!([ss_init["Y"]], color=:gray, ls=:dot, label="SS Inicial")
    hline!([ss_fin["Y"]], color=:black, ls=:dash, label="SS Final")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Producción per cápita")
    xlabel!("Periodos")
    ylabel!("Y")
    
    p2 = plot(t_axis, res["C"], color=:purple, lw=2.5, label="Consumo (C)")
    hline!([ss_init["C"]], color=:gray, ls=:dot, label="SS Inicial")
    hline!([ss_fin["C"]], color=:black, ls=:dash, label="SS Final")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Consumo per cápita")
    xlabel!("Periodos")
    ylabel!("C")
    
    p3 = plot(t_axis, res["I"], color=:orange, lw=2.5, label="Inversión (I)")
    hline!([ss_init["I"]], color=:gray, ls=:dot, label="SS Inicial")
    hline!([ss_fin["I"]], color=:black, ls=:dash, label="SS Final")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Inversión per cápita")
    xlabel!("Periodos")
    ylabel!("I")
    
    p4 = plot(t_axis, res["K"], color=:forestgreen, lw=2.5, label="Capital (K)")
    hline!([ss_init["K"]], color=:gray, ls=:dot, label="SS Inicial")
    hline!([ss_fin["K"]], color=:black, ls=:dash, label="SS Final")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Capital per cápita")
    xlabel!("Periodos")
    ylabel!("K")
    
    plot(p1, p2, p3, p4, layout=(2,2), size=(900, 600), 
         plot_title="Ajuste Dinámico frente a Shock Permanente (Ramsey)", top_margin=10mm)
end


## 4. Precisión de Resolvedores: Blanchard-Khan frente a Solución No Lineal

El resolvedor de Blanchard-Khan asume una linealización de primer orden alrededor del estado estacionario y es altamente preciso para shocks de baja envergadura. Para shocks de gran tamaño, sin embargo, el error acumulado por la curvatura puede ser relevante.

A continuación, compara las trayectorias calculadas mediante **Blanchard-Khan (BK)** y por el resolvedor **No Lineal Exacto (SciPy fsolve)** ante perturbaciones de distinta envergadura.


In [ ]:
# Comparación Lineal (Blanchard-Kahn) vs No Lineal (Shooting)
@manipulate for A_shock in 0.70:0.02:1.30
    
    params = default_calibration(RamseyParams)
    ss_comp = compute_ramsey_steady_state(params)
    K0 = ss_comp["K"]
    T_sim = 80
    t_shock = 5
    
    A_path = fill(1.00, T_sim)
    A_path[t_shock+1:end] .= A_shock
    
    n_path = fill(params.n, T_sim)
    
    # Resolver
    res_lin = solve_ramsey_linearized(params, K0, A_shock, params.n, params.beta, T_sim, t_shock)
    res_nonlin = solve_ramsey_nonlinear(params, K0, A_path, n_path, T_sim, t_shock)
    
    # Calcular errores relativos máximos
    err_C = maximum(abs.(res_nonlin["C"] .- res_lin["C"])) / ss_comp["C"] * 100
    err_K = maximum(abs.(res_nonlin["K"] .- res_lin["K"])) / ss_comp["K"] * 100
    println("Error relativo máximo en Consumo (C): ", round(err_C, digits=4), "%")
    println("Error relativo máximo en Capital (K): ", round(err_K, digits=4), "%")
    
    t_axis = 0:(T_sim - 1)
    
    p1 = plot(t_axis, res_nonlin["C"], color=:purple, lw=3, label="Exacto No Lineal")
    plot!(t_axis, res_lin["C"], color=:purple, ls=:dash, lw=2, label="Blanchard-Khan (Lineal)")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Consumo (C_t)")
    xlabel!("Periodos")
    ylabel!("C")
    
    p2 = plot(t_axis, res_nonlin["K"], color=:forestgreen, lw=3, label="Exacto No Lineal")
    plot!(t_axis, res_lin["K"], color=:forestgreen, ls=:dash, lw=2, label="Blanchard-Khan (Lineal)")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Capital (K_t)")
    xlabel!("Periodos")
    ylabel!("K")
    
    plot(p1, p2, layout=(1,2), size=(800, 350), 
         plot_title="Comparación Lineal vs No Lineal (Shock A=$(A_shock))", top_margin=10mm)
end


## 5. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones tras interactuar con el simulador del modelo de crecimiento óptimo de Ramsey:

1.  **Dinámica de Consumo y Ahorro en Ramsey (Comparación con Solow-Swan)**:
    *   Simula un incremento permanente de la TFP del $5\%$ ($A = 1.05$, Sección 3). Describe cómo responden el consumo y la inversión en el momento del impacto ($t=5$). ¿Por qué el consumo salta instantáneamente en Ramsey, en lugar de mantenerse constante como ocurriría bajo una tasa de ahorro fija?
    *   Explica qué motiva a los hogares representativos a aumentar su inversión y sacrificar consumo relativo a largo plazo para acumular más capital.
2.  **Shock de Paciencia (Preferencia Intertemporal $\beta$)**:
    *   Simula un incremento del factor de descuento $\beta$ de $0.97$ a $0.98$ (Sección 3). Describe el comportamiento del consumo y la inversión en el período del shock $t=5$. ¿Por qué se produce una caída instantánea del consumo? ¿Cómo compensa este "sacrificio" de corto plazo al bienestar dinámico de los consumidores en el largo plazo?
3.  **Límites de la linealización**:
    *   En la Sección 4, establece un shock de TFP estándar del $5\%$ ($A = 1.05$) y anota el error relativo máximo entre el resolvedor de Blanchard-Khan y el no lineal exacto.
    *   Aplica un shock masivo del $30\%$ ($A = 1.30$). ¿Cómo afecta esto al error relativo en consumo y capital? Explica la relación entre la magnitud de la perturbación respecto al estado estacionario base y la validez de las aproximaciones de primer orden.


## 6. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Aislamiento del Backend Computacional**: Las rutinas paramétricas, el cálculo de estado estacionario y los resolvedores dinámicos están aislados en el módulo [`ramsey.py`](file:///c:/Users/AntonioRC/Desktop/PIE/src/macroaicomp/models/ramsey.py), manteniendo el notebook limpio y enfocado exclusivamente en la didáctica y visualización.
2.  **Inicialización Inteligente (Smart Guessing)**: El resolvedor no lineal exacto (`fsolve`) utiliza la trayectoria de Blanchard-Khan linealizada como punto de partida, garantizando una convergencia instantánea y robusta.
3.  **Higiene del Repositorio**: El cuaderno se acoge a las directivas de control de versiones del repo limpiando salidas volátiles mediante `nbstripout` en `pre-commit`.


## 7. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [ ]:
# Benchmark simulation para No Lineal y Blanchard-Kahn
A_bench = fill(1.05, 50)
A_bench[1] = 1.00
n_bench = fill(params_base.n, 50)

println("Benchmark NLsolve (Shooting No Lineal):")
@btime solve_ramsey_nonlinear($params_base, $ss["K"], $A_bench, $n_bench, 50, 1)

println("Benchmark Blanchard-Kahn (Lineal):")
@btime solve_ramsey_linearized($params_base, $ss["K"], 1.05, $params_base.n, $params_base.beta, 50, 1)
